# LeetCode 388 — Longest Absolute File Path bằng Python

Link đề: https://leetcode.com/problems/longest-absolute-file-path/ · Độ khó: **Medium** · Chủ đề: Stack, Depth-First Search, String

Skeleton LeetCode:

```python
class Solution:
    def lengthLongestPath(self, input: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cách **một chuỗi duy nhất** (có ký tự xuống dòng `\n` và tab `\t`) mô tả cả một **cây thư mục**: số tab đầu dòng = **độ sâu** của tên đó.
2. Nắm các công cụ Python cần dùng: `split('\n')`, `lstrip('\t')`, `in` để tìm dấu chấm, `list` làm **ngăn xếp (stack)** với `append` / `pop` / `[-1]`.
3. Đi từ cách **vét cạn** (với mỗi file, lần ngược lên để dựng đường dẫn) → **ngăn xếp tên** (mô phỏng lệnh `cd`) → **ngăn xếp độ dài** (không cần giữ tên) → **mảng theo độ sâu** (không cần `pop`).
4. Hiểu mấu chốt: đầu vào là một **duyệt DFS theo thứ tự** của cây, nên chỉ cần nhớ **chuỗi tổ tiên hiện tại** — và để tính độ dài, chỉ cần nhớ **độ dài tích lũy** của từng tổ tiên.
5. Tránh các lỗi kinh điển: quên `+1` cho dấu `/`, dùng `strip()` làm mất dấu cách trong tên, quên bỏ bớt ngăn xếp khi lên cấp, cập nhật đáp án ở thư mục thay vì file.

## 2. Đề bài & ví dụ

Một hệ thống file có **thư mục** và **file**. Ta biểu diễn nó bằng một chuỗi: mỗi dòng là một tên, **số ký tự tab ở đầu dòng** cho biết độ sâu (0 = nằm ở gốc). Ví dụ (ký hiệu `⟶` là tab):

```
dir
⟶ subdir1
⟶ ⟶ file1.ext
⟶ ⟶ subsubdir1
⟶ subdir2
⟶ ⟶ subsubdir2
⟶ ⟶ ⟶ file2.ext
```

Viết thành chuỗi Python: `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"`.

**Đường dẫn tuyệt đối** của một file là các thư mục phải mở theo thứ tự, nối nhau bằng `/`. Ví dụ file `file2.ext` có đường dẫn `dir/subdir2/subsubdir2/file2.ext` (độ dài 32). Tên thư mục gồm chữ, số, dấu cách (không có dấu chấm); tên file có dạng `ten.duoi` (cả hai phần gồm chữ, số, dấu cách).

Trả về **độ dài đường dẫn tuyệt đối dài nhất tới một FILE**. Nếu không có file nào, trả về `0`.

**Ví dụ 1:** `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` → `20` (đường dẫn `dir/subdir2/file.ext`).

**Ví dụ 2:** chuỗi dài ở trên → `32` (hai file: `dir/subdir1/file1.ext` dài 21 và `dir/subdir2/subsubdir2/file2.ext` dài 32).

**Ví dụ 3:** `"a"` → `0` (chỉ có một thư mục, không có file).

**Ràng buộc:** `1 <= input.length <= 10^4`; `input` chỉ gồm chữ cái, `\n`, `\t`, `.`, dấu cách và chữ số; mọi tên đều có độ dài dương; hệ thống file luôn hợp lệ.

In [1]:
examples = [
    ("dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext", 20),
    ("dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext", 32),
    ("a", 0),
]
for text, expected in examples:
    print(repr(text), "-> kỳ vọng", expected)

'dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext' -> kỳ vọng 20
'dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext' -> kỳ vọng 32
'a' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Ký tự đặc biệt `\n` và `\t`

Trong chuỗi Python, `\n` là **một** ký tự xuống dòng, `\t` là **một** ký tự tab (chiều dài chuỗi tính mỗi cái là `1`). `print` hiển thị chúng như khoảng trắng/xuống dòng, còn `repr` cho ta thấy dạng "thô".

In [2]:
s = "dir\n\tsub"
print(s)                 # in ra 2 dòng, dòng 2 thụt vào
print(repr(s))           # 'dir\n\tsub'
print(len(s))            # 3 + 1 + 1 + 3 = 8  (\n và \t mỗi cái chỉ tính 1)

dir
	sub
'dir\n\tsub'
8


### 3.2. `split('\n')` — cắt chuỗi thành các dòng

`s.split('\n')` cắt chuỗi tại mỗi ký tự `\n` và trả về **list các dòng** (bỏ chính ký tự `\n`). Đề đảm bảo không có tên rỗng nên không có dòng rỗng.

In [3]:
text = "dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"
lines = text.split('\n')
print(lines)             # ['dir', '\tsubdir1', '\tsubdir2', '\t\tfile.ext']
print(len(lines))        # 4

['dir', '\tsubdir1', '\tsubdir2', '\t\tfile.ext']
4


### 3.3. `lstrip('\t')` và độ sâu

Số tab đầu dòng = độ sâu. Cách gọn: `name = line.lstrip('\t')` bỏ **chỉ các tab ở đầu**; `depth = len(line) - len(name)`. Lưu ý: phải truyền rõ `'\t'`. Nếu gọi `lstrip()` không đối số, Python bỏ **mọi khoảng trắng** (kể cả dấu cách thuộc về tên!) — đây là lỗi hay gặp (xem mục 13).

In [4]:
line = "\t\tfile.ext"
name = line.lstrip('\t')
depth = len(line) - len(name)
print(repr(name), depth)         # 'file.ext' 2

line2 = "\t a b"                 # tên là " a b" (bắt đầu bằng dấu cách)
print(repr(line2.lstrip('\t')))  # ' a b'  (giữ dấu cách)
print(repr(line2.lstrip()))      # 'a b'   (mất dấu cách của tên!)

'file.ext' 2
' a b'
'a b'


### 3.4. Nhận biết file: có dấu chấm `.`

Đề cho: tên file dạng `name.extension` còn tên thư mục **không có dấu chấm**. Vì vậy `'.' in name` cho biết đó là file. Toán tử `in` với chuỗi kiểm tra *chuỗi con*.

In [5]:
for name in ["dir", "file.ext", "my file.txt", "sub dir 2"]:
    print(f"{name!r:14} is_file = {'.' in name}")

'dir'          is_file = False
'file.ext'     is_file = True
'my file.txt'  is_file = True
'sub dir 2'    is_file = False


### 3.5. `list` làm ngăn xếp (stack)

**Stack** là kiểu "vào sau — ra trước" (LIFO), như chồng đĩa. Trong Python dùng `list`: `append(x)` đặt lên **đỉnh**, `pop()` lấy **đỉnh** ra, `stack[-1]` **xem** đỉnh mà không lấy, `len(stack)` là số phần tử. Cả bốn thao tác đều `O(1)`.

In [6]:
stack = []
stack.append("dir")
stack.append("subdir1")
print(stack, "đỉnh =", stack[-1], "kích thước =", len(stack))   # ['dir', 'subdir1'] đỉnh = subdir1 kích thước = 2
print(stack.pop())                                               # subdir1
print(stack)                                                     # ['dir']
while len(stack) > 0:        # bỏ bớt cho đến khi chỉ còn 0 phần tử
    stack.pop()
print(stack)                                                     # []

['dir', 'subdir1'] đỉnh = subdir1 kích thước = 2
subdir1
['dir']
[]


### 3.6. Độ dài đường dẫn = tổng độ dài tên + số dấu `/`

Đường dẫn `dir/subdir2/file.ext` gồm 3 tên (`3 + 7 + 8 = 18` ký tự) và 2 dấu `/` → `20`. Tổng quát: `k` tên cần `k − 1` dấu `/`. Do đó `len('/'.join(parts)) == sum(len(p) for p in parts) + len(parts) - 1`.

In [7]:
parts = ["dir", "subdir2", "file.ext"]
path = "/".join(parts)
print(path, len(path))                                   # dir/subdir2/file.ext 20
print(sum(len(p) for p in parts) + len(parts) - 1)       # 20

dir/subdir2/file.ext 20
20


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy "đóng vai" trình duyệt file và đọc từng dòng của ví dụ 2, theo dõi **ta đang đứng ở thư mục nào** (chuỗi tổ tiên):

| Dòng | Độ sâu | Tên | Ta đang đứng ở (các tổ tiên) | Loại |
|---|---|---|---|---|
| `dir` | 0 | dir | (gốc) | thư mục |
| `\tsubdir1` | 1 | subdir1 | `dir` | thư mục |
| `\t\tfile1.ext` | 2 | file1.ext | `dir/subdir1` | **file** → `dir/subdir1/file1.ext` = 21 |
| `\t\tsubsubdir1` | 2 | subsubdir1 | `dir/subdir1` | thư mục |
| `\tsubdir2` | 1 | subdir2 | `dir` (đã lùi ra khỏi `subdir1`) | thư mục |
| `\t\tsubsubdir2` | 2 | subsubdir2 | `dir/subdir2` | thư mục |
| `\t\t\tfile2.ext` | 3 | file2.ext | `dir/subdir2/subsubdir2` | **file** → 32 |

Quy tắc rút ra: một mục ở **độ sâu `d`** có **đúng `d` tổ tiên**, và đó là `d` thư mục gần nhất đang "mở". Khi gặp dòng nông hơn, ta **lùi ra** (bỏ bớt tổ tiên) cho đến khi còn `d` tổ tiên. Đây chính là hành vi của một **ngăn xếp**.

In [8]:
def parse(text):
    """Trả về list (độ_sâu, tên) cho từng dòng."""
    result = []
    for line in text.split('\n'):
        name = line.lstrip('\t')
        result.append((len(line) - len(name), name))
    return result


for d, name in parse(examples[1][0]):
    print(d, name)

0 dir
1 subdir1
2 file1.ext
2 subsubdir1
1 subdir2
2 subsubdir2
3 file2.ext


## 5. Cách 1 — Brute Force: với mỗi file, lần ngược lên tìm tổ tiên

### Ý tưởng

1. Tách input thành các dòng `(độ_sâu, tên)` như hàm `parse` ở trên.
2. Với **mỗi file** ở vị trí `i`, độ sâu `d`: ta cần các tổ tiên ở độ sâu `d−1, d−2, ..., 0`. Duyệt **ngược** từ dòng `i−1` lên đầu; dòng đầu tiên gặp có độ sâu đúng bằng "độ sâu cần tìm" chính là tổ tiên tương ứng (vì đầu vào theo thứ tự DFS, tổ tiên gần nhất luôn nằm trước và gần nhất).
3. Dựng đường dẫn đầy đủ bằng `'/'.join(...)`, lấy độ dài, giữ giá trị lớn nhất.

Cách này rất trực giác (làm đúng như định nghĩa đề) nhưng mỗi file có thể phải quét lại gần hết các dòng phía trước.

In [9]:
def longest_path_brute(text):
    items = parse(text)
    best = 0
    for i, (d, name) in enumerate(items):
        if '.' not in name:
            continue                      # chỉ xét file
        parts = [name]
        need = d - 1                      # độ sâu của tổ tiên cần tìm tiếp
        j = i - 1
        while need >= 0:
            if items[j][0] == need:
                parts.append(items[j][1])
                need -= 1
            j -= 1
        path = '/'.join(reversed(parts))
        best = max(best, len(path))
    return best


for text, expected in examples:
    print(longest_path_brute(text), expected)

20 20
32 32
0 0


### Độ phức tạp & nút thắt

- Gọi `L` là số dòng, `n` là độ dài chuỗi. Mỗi file có thể quét ngược tới `O(L)` dòng, rồi `join` tốn `O(độ dài đường dẫn)`. Tệ nhất `O(L · n)` (khoảng `O(n²)`), bộ nhớ `O(n)` cho mỗi đường dẫn dựng ra.
- Với `n = 10^4` vẫn chạy được, nhưng **lãng phí**: ta dựng lại cả chuỗi đường dẫn chỉ để lấy *độ dài*, và quét ngược lặp đi lặp lại những tổ tiên giống nhau (hai file cùng thư mục chia sẻ chung tổ tiên).

**Nút thắt:** tổ tiên của các file liên tiếp gần như giống nhau → hãy **nhớ lại chuỗi tổ tiên** thay vì tìm lại.

## 6. Ý tưởng thuật toán / bước đột phá

Đọc dòng theo thứ tự, ta chỉ cần giữ **chuỗi tổ tiên hiện tại**. Khi gặp dòng ở độ sâu `d`:

1. **Bỏ bớt** cuối chuỗi tổ tiên cho đến khi còn đúng `d` phần tử (ta đã "đi ra" khỏi các thư mục sâu hơn).
2. Nếu dòng là **file**: đường dẫn = các tổ tiên + file → cập nhật đáp án.
3. Nếu là **thư mục**: **đẩy** nó vào chuỗi tổ tiên (từ giờ nó là tổ tiên của những dòng sâu hơn).

Bước đột phá thứ hai: để tính *độ dài* ta **không cần giữ tên**, chỉ cần giữ **độ dài tích lũy** của đường dẫn tới từng tổ tiên. Nếu đỉnh ngăn xếp là `P` (độ dài đường dẫn tới thư mục cha, không có `/` cuối) thì một mục có tên dài `m` có đường dẫn dài `P + 1 + m`. Chỉ một con số mỗi tổ tiên!

In [10]:
# Quan sát ngăn xếp độ dài trên ví dụ 2, in sau mỗi dòng
stack = []
for d, name in parse(examples[1][0]):
    while len(stack) > d:
        stack.pop()
    cur = len(name) + (stack[-1] + 1 if stack else 0)
    kind = "file" if '.' in name else "dir "
    if kind == "dir ":
        stack.append(cur)
    print(f"{kind} {name:12} d={d} cur={cur:2}  stack={stack}")

dir  dir          d=0 cur= 3  stack=[3]
dir  subdir1      d=1 cur=11  stack=[3, 11]
file file1.ext    d=2 cur=21  stack=[3, 11]
dir  subsubdir1   d=2 cur=22  stack=[3, 11, 22]
dir  subdir2      d=1 cur=11  stack=[3, 11]
dir  subsubdir2   d=2 cur=22  stack=[3, 11, 22]
file file2.ext    d=3 cur=32  stack=[3, 11, 22]


## 7. Các cách tiếp theo

Ta có tổng cộng **4 cách**: (1) brute force lần ngược; (2) ngăn xếp tên (mô phỏng `cd`); (3) ngăn xếp độ dài; (4) mảng độ dài theo độ sâu (không cần `pop`).

### Cách 2 — Ngăn xếp tên (mô phỏng lệnh `cd`)

Giải thích từng đoạn:

- `path = []` là chuỗi tên các thư mục tổ tiên.
- Với mỗi dòng: `del path[d:]` (hoặc `pop` lặp) để chỉ còn đúng `d` tổ tiên.
- Nếu là file: `'/'.join(path + [name])` rồi lấy `len`. Nếu là thư mục: `path.append(name)`.

Dễ hiểu nhất, nhưng mỗi file vẫn `join` cả đường dẫn nên tốn `O(độ sâu)` thời gian cho một file.

In [11]:
def longest_path_names(text):
    path = []
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        del path[d:]                         # lùi ra: chỉ giữ d tổ tiên
        if '.' in name:
            best = max(best, len('/'.join(path + [name])))
        else:
            path.append(name)
    return best


for text, expected in examples:
    print(longest_path_names(text), expected)

20 20
32 32
0 0


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionNames:
    def lengthLongestPath(self, input: str) -> int:
        path = []
        best = 0
        for line in input.split('\n'):
            name = line.lstrip('\t')
            d = len(line) - len(name)
            del path[d:]
            if '.' in name:
                best = max(best, len('/'.join(path + [name])))
            else:
                path.append(name)
        return best


print([SolutionNames().lengthLongestPath(t) for t, _ in examples])

[20, 32, 0]


Độ phức tạp: mỗi file `join` tốn `O(độ dài đường dẫn)` ≤ `O(n)`, tệ nhất `O(n²)` nếu có nhiều file ở sâu; bộ nhớ `O(n)`. Chạy được ở `n ≤ 10^4` nhưng vẫn chưa tối ưu.

### Cách 3 — Ngăn xếp độ dài (cách chuẩn)

Từ nhận xét ở mục 6: thay tên bằng **độ dài tích lũy**.

- `stack[-1]` = độ dài đường dẫn tới thư mục cha gần nhất (không tính `/` cuối).
- Một mục tên dài `m`: `cur = m` nếu stack rỗng, ngược lại `cur = stack[-1] + 1 + m`.
- Thư mục: `stack.append(cur)`. File: `best = max(best, cur)`.

Mỗi dòng vào/ra stack tối đa một lần ⇒ tổng `O(n)`.

In [13]:
def longest_path_stack(text):
    stack = []                               # stack[k] = độ dài đường dẫn tới tổ tiên ở độ sâu k
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        while len(stack) > d:                # lùi ra khỏi các thư mục sâu hơn
            stack.pop()
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        if '.' in name:
            best = max(best, cur)
        else:
            stack.append(cur)
    return best


for text, expected in examples:
    print(longest_path_stack(text), expected)

20 20
32 32
0 0


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionStack:
    def lengthLongestPath(self, input: str) -> int:
        stack = []
        best = 0
        for line in input.split('\n'):
            name = line.lstrip('\t')
            d = len(line) - len(name)
            while len(stack) > d:
                stack.pop()
            cur = len(name) + (stack[-1] + 1 if stack else 0)
            if '.' in name:
                best = max(best, cur)
            else:
                stack.append(cur)
        return best


print([SolutionStack().lengthLongestPath(t) for t, _ in examples])

[20, 32, 0]


Độ phức tạp: **Time `O(n)`** (duyệt từng ký tự một lần để split/lstrip; mỗi dòng push/pop `O(1)`), **Space `O(n)`** (stack tối đa bằng độ sâu ≤ số dòng).

### Cách 4 — Mảng độ dài theo độ sâu (không cần `pop`)

Nhận xét: khi vào một dòng ở độ sâu `d`, ta chỉ quan tâm "độ dài tiền tố tại độ sâu `d`". Gọi `prefix[d]` = **độ dài của đường dẫn tới thư mục chứa mục ở độ sâu `d`, đã tính cả dấu `/` cuối** (`prefix[0] = 0` vì ở gốc chưa có gì).

- Mục ở độ sâu `d`, tên dài `m`: đường dẫn dài `prefix[d] + m`.
- Nếu là thư mục: `prefix[d + 1] = prefix[d] + m + 1` — **ghi đè** giá trị cũ (nếu có).

Vì đầu vào theo thứ tự DFS, mỗi lần ta đọc một dòng ở độ sâu `d`, giá trị `prefix[d]` đã được ghi bởi **thư mục cha gần nhất** — những giá trị cũ ở độ sâu sâu hơn không còn ai dùng đến nên **không cần xóa**. Dùng `dict` (hoặc list đủ lớn) thay stack, không có `pop`.

In [15]:
def longest_path_levels(text):
    prefix = {0: 0}                          # prefix[d]: độ dài tiền tố (kể cả '/') cho mục ở độ sâu d
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        m = len(name)
        if '.' in name:
            best = max(best, prefix[d] + m)
        else:
            prefix[d + 1] = prefix[d] + m + 1
    return best


for text, expected in examples:
    print(longest_path_levels(text), expected)

20 20
32 32
0 0


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionLevels:
    def lengthLongestPath(self, input: str) -> int:
        prefix = {0: 0}
        best = 0
        for line in input.split('\n'):
            name = line.lstrip('\t')
            d = len(line) - len(name)
            if '.' in name:
                best = max(best, prefix[d] + len(name))
            else:
                prefix[d + 1] = prefix[d] + len(name) + 1
        return best


print([SolutionLevels().lengthLongestPath(t) for t, _ in examples])

[20, 32, 0]


Độ phức tạp: **Time `O(n)`**, **Space `O(D)`** với `D` là độ sâu tối đa (≤ số dòng). Code ngắn nhất và không có vòng `while` bên trong.

## 8. TRACE từng bước

### Bảng trạng thái cho Cách 3 (ngăn xếp độ dài) trên ví dụ 2

| Dòng | `d` | `name` | `stack` sau khi bỏ bớt | `cur` | Hành động | `stack` sau | `best` |
|---|---|---|---|---|---|---|---|
| `dir` | 0 | dir | `[]` | 3 | thư mục → push | `[3]` | 0 |
| `\tsubdir1` | 1 | subdir1 | `[3]` | 3+1+7 = 11 | push | `[3, 11]` | 0 |
| `\t\tfile1.ext` | 2 | file1.ext | `[3, 11]` | 11+1+9 = 21 | file → cập nhật | `[3, 11]` | 21 |
| `\t\tsubsubdir1` | 2 | subsubdir1 | `[3, 11]` | 11+1+10 = 22 | push | `[3, 11, 22]` | 21 |
| `\tsubdir2` | 1 | subdir2 | `[3]` (pop 2 lần) | 3+1+7 = 11 | push | `[3, 11]` | 21 |
| `\t\tsubsubdir2` | 2 | subsubdir2 | `[3, 11]` | 22 | push | `[3, 11, 22]` | 21 |
| `\t\t\tfile2.ext` | 3 | file2.ext | `[3, 11, 22]` | 22+1+9 = 32 | file → cập nhật | `[3, 11, 22]` | **32** |

Chú ý dòng `\tsubdir2`: stack đang có 3 phần tử nhưng độ sâu là 1 nên **bỏ 2 phần tử** đến khi còn 1 (`[3]`) — đây là lúc ta "lùi ra khỏi `subdir1`".

In [17]:
def trace_stack(text):
    stack = []
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        before = list(stack)
        while len(stack) > d:
            stack.pop()
        popped = len(before) - len(stack)
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        if '.' in name:
            best = max(best, cur)
            action = "FILE -> best"
        else:
            stack.append(cur)
            action = "DIR  -> push"
        print(f"{name!r:14} d={d} pop {popped} lần  cur={cur:2}  {action:13} stack={stack} best={best}")
    return best


print("kết quả:", trace_stack(examples[1][0]))

'dir'          d=0 pop 0 lần  cur= 3  DIR  -> push  stack=[3] best=0
'subdir1'      d=1 pop 0 lần  cur=11  DIR  -> push  stack=[3, 11] best=0
'file1.ext'    d=2 pop 0 lần  cur=21  FILE -> best  stack=[3, 11] best=21
'subsubdir1'   d=2 pop 0 lần  cur=22  DIR  -> push  stack=[3, 11, 22] best=21
'subdir2'      d=1 pop 2 lần  cur=11  DIR  -> push  stack=[3, 11] best=21
'subsubdir2'   d=2 pop 0 lần  cur=22  DIR  -> push  stack=[3, 11, 22] best=21
'file2.ext'    d=3 pop 0 lần  cur=32  FILE -> best  stack=[3, 11, 22] best=32
kết quả: 32


### Bảng cho Cách 4 (mảng theo độ sâu)

Cùng ví dụ, `prefix` ban đầu `{0: 0}` (ở đây `prefix[d]` đã gồm dấu `/` cuối):

| Dòng | `d` | `m` | Hành động | `prefix` sau | `best` |
|---|---|---|---|---|---|
| `dir` | 0 | 3 | thư mục: `prefix[1] = 0+3+1` | `{0:0, 1:4}` | 0 |
| `subdir1` | 1 | 7 | `prefix[2] = 4+7+1` | `{0:0, 1:4, 2:12}` | 0 |
| `file1.ext` | 2 | 9 | file: `12+9 = 21` | giữ nguyên | 21 |
| `subsubdir1` | 2 | 10 | `prefix[3] = 12+10+1` | `{..., 3:23}` | 21 |
| `subdir2` | 1 | 7 | **ghi đè** `prefix[2] = 4+7+1` | `{..., 2:12, 3:23}` | 21 |
| `subsubdir2` | 2 | 10 | **ghi đè** `prefix[3] = 12+10+1` | `{..., 3:23}` | 21 |
| `file2.ext` | 3 | 9 | file: `23+9 = 32` | giữ nguyên | **32** |

In [18]:
def trace_levels(text):
    prefix = {0: 0}
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        m = len(name)
        if '.' in name:
            best = max(best, prefix[d] + m)
            print(f"{name!r:14} d={d} m={m:2} FILE  {prefix[d]}+{m} = {prefix[d] + m:2}  best={best}")
        else:
            prefix[d + 1] = prefix[d] + m + 1
            print(f"{name!r:14} d={d} m={m:2} DIR   prefix[{d + 1}] = {prefix[d + 1]:2}  prefix={prefix}")
    return best


print("kết quả:", trace_levels(examples[1][0]))

'dir'          d=0 m= 3 DIR   prefix[1] =  4  prefix={0: 0, 1: 4}
'subdir1'      d=1 m= 7 DIR   prefix[2] = 12  prefix={0: 0, 1: 4, 2: 12}
'file1.ext'    d=2 m= 9 FILE  12+9 = 21  best=21
'subsubdir1'   d=2 m=10 DIR   prefix[3] = 23  prefix={0: 0, 1: 4, 2: 12, 3: 23}
'subdir2'      d=1 m= 7 DIR   prefix[2] = 12  prefix={0: 0, 1: 4, 2: 12, 3: 23}
'subsubdir2'   d=2 m=10 DIR   prefix[3] = 23  prefix={0: 0, 1: 4, 2: 12, 3: 23}
'file2.ext'    d=3 m= 9 FILE  23+9 = 32  best=32
kết quả: 32


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (ngăn xếp độ dài) làm bản nộp chính: dễ giải thích, `O(n)`.

```python
class Solution:
    def lengthLongestPath(self, input: str) -> int:
        stack = []
        best = 0
        for line in input.split('\n'):
            name = line.lstrip('\t')
            d = len(line) - len(name)
            while len(stack) > d:
                stack.pop()
            cur = len(name) + (stack[-1] + 1 if stack else 0)
            if '.' in name:
                best = max(best, cur)
            else:
                stack.append(cur)
        return best
```

In [19]:
class Solution:
    def lengthLongestPath(self, input: str) -> int:
        stack = []
        best = 0
        for line in input.split('\n'):
            name = line.lstrip('\t')
            d = len(line) - len(name)
            while len(stack) > d:
                stack.pop()
            cur = len(name) + (stack[-1] + 1 if stack else 0)
            if '.' in name:
                best = max(best, cur)
            else:
                stack.append(cur)
        return best


for text, expected in examples:
    got = Solution().lengthLongestPath(text)
    print(got, expected, got == expected)

20 20 True
32 32 True
0 0 True


## 10. Vì sao đúng

**Bất biến:** sau khi xử lý xong phần "bỏ bớt" của một dòng ở độ sâu `d`, `stack` có **đúng `d` phần tử**, và phần tử thứ `k` (`k = 0..d−1`) là độ dài đường dẫn tới **tổ tiên thật sự ở độ sâu `k`** của dòng hiện tại.

Chứng minh bằng quy nạp theo dòng:

- Dòng đầu tiên có độ sâu 0 (đề đảm bảo hợp lệ), stack rỗng: đúng.
- Giả sử đúng trước dòng thứ `t`. Dòng `t−1` ở độ sâu `d'`. Sau khi xử lý, nếu nó là thư mục, stack có `d'+1` phần tử (đúng cho con của nó); nếu là file, stack có `d'` phần tử. Dòng `t` hợp lệ nên độ sâu `d ≤ d' + 1` (nếu `t−1` là thư mục) hoặc `d ≤ d'` (nếu là file). Ta bỏ bớt cho đến còn `d` phần tử; `d` phần tử đầu của stack chính là các tổ tiên ở độ sâu `0..d−1` (các thư mục ở độ sâu nông hơn không bị đụng tới) ⇒ bất biến giữ.

Với bất biến đó, `cur = len(name) + 1 + stack[-1]` đúng bằng độ dài đường dẫn tuyệt đối của dòng hiện tại (tổng độ dài các tên tổ tiên, các dấu `/` giữa chúng, thêm `/` và tên). Ta chỉ lấy `max` ở **file** vì đề hỏi đường dẫn tới file; thư mục chỉ để dựng tiền tố. Nếu không có file nào, `best` giữ giá trị `0` ban đầu — đúng yêu cầu.

## 11. Time/Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute (lần ngược) | `O(L · n)` ≈ `O(n²)` | `O(n)` |
| 2. Ngăn xếp tên | `O(n²)` tệ nhất (mỗi file `join`) | `O(n)` |
| 3. Ngăn xếp độ dài | **`O(n)`** | `O(n)` |
| 4. Mảng theo độ sâu | **`O(n)`** | `O(D)` |

`n` = độ dài chuỗi, `L` = số dòng, `D` = độ sâu tối đa. Ở Cách 3/4, `split` và `lstrip` mỗi ký tự được chạm vài lần, mỗi dòng push/pop một lần ⇒ tổng `O(n)`.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

In [20]:
all_methods = [longest_path_brute, longest_path_names, longest_path_stack, longest_path_levels,
               SolutionNames().lengthLongestPath, SolutionStack().lengthLongestPath,
               SolutionLevels().lengthLongestPath, Solution().lengthLongestPath]

cases = [
    ("dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext", 20),
    ("dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext", 32),
    ("a", 0),                                    # chỉ có thư mục
    ("a.b", 3),                                  # một file ở gốc
    ("a\n\tb\n\t\tc", 0),                        # chuỗi thư mục, không file
    ("dir\n\tfile name.txt", len("dir/file name.txt")),   # tên file có dấu cách
    ("my dir\n\tmy file.t x", len("my dir/my file.t x")),   # cả hai có dấu cách
    ("a.b\nc.d", 3),                             # hai file ở gốc: đường dẫn không nối nhau
    ("a\nbb.c\nccc", 4),                         # file ở gốc sau một thư mục
    ("a\n\tbbbbbb\n\tc.d", 1 + 1 + 3),            # thư mục anh em dài hơn không được tính
    ("a\n\tb\n\t\tc.d\ne.f", 7),                 # a/b/c.d = 7 > e.f = 3
    ("a\n\tb\n\t\tc.d\nx.yyyyyyy", 9),           # file ở gốc dài hơn
]
for text, expected in cases:
    for f in all_methods:
        assert f(text) == expected, (f.__name__ if hasattr(f, '__name__') else f, text, expected, f(text))
print("OK:", len(cases), "test cố định x", len(all_methods), "cách")

OK: 12 test cố định x 8 cách


### 12.2. Phản ví dụ cho các ý tưởng sai

Một đáp án "gần đúng" hay gặp: lấy **độ dài của dòng dài nhất** hoặc **tổng độ dài mọi tên**. Các phản ví dụ dưới đây cho thấy chúng sai:

In [21]:
def wrong_sum_all_names(text):               # SAI: cộng tên của mọi dòng
    return sum(len(l.lstrip('\t')) for l in text.split('\n') if '.' in l)

def wrong_longest_line(text):                # SAI: dòng dài nhất, không kể tổ tiên
    return max((len(l.lstrip('\t')) for l in text.split('\n') if '.' in l), default=0)

t = "dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"
print(wrong_sum_all_names(t), wrong_longest_line(t), "đúng là", Solution().lengthLongestPath(t))

18 9 đúng là 32


### 12.3. Sinh dữ liệu ngẫu nhiên với oracle độc lập

Ta tự sinh một **cây thư mục thật** (list lồng nhau) rồi:
- in ra thành chuỗi theo DFS (có tab);
- tính đáp án bằng cách **duyệt cây và nối chuỗi đường dẫn thật** `'/'.join(...)` — hoàn toàn độc lập với việc xử lý chuỗi ở trên.

In [22]:
import random

ALPHA = "abcXY 019"                                  # có cả dấu cách và chữ số


def rand_name(rng, is_file):
    base = "".join(rng.choice(ALPHA) for _ in range(rng.randint(1, 5)))
    if is_file:
        ext = "".join(rng.choice(ALPHA) for _ in range(rng.randint(1, 3)))
        return base + "." + ext
    return base


def gen_dir(rng, depth, max_depth):
    """Một thư mục: (tên, [con...]); con là ('f', tên) hoặc thư mục."""
    children = []
    for _ in range(rng.randint(0, 3)):
        if depth < max_depth and rng.random() < 0.5:
            children.append(gen_dir(rng, depth + 1, max_depth))
        else:
            children.append(('f', rand_name(rng, True)))
    return (rand_name(rng, False), children)


def render(node, depth, out):
    if node[0] == 'f':
        out.append("\t" * depth + node[1])
    else:
        out.append("\t" * depth + node[0])
        for ch in node[1]:
            render(ch, depth + 1, out)


def oracle(node, prefix):
    """Duyệt cây, nối đường dẫn thật, trả về độ dài lớn nhất tới file."""
    if node[0] == 'f':
        return len(prefix + node[1])
    best = 0
    for ch in node[1]:
        best = max(best, oracle(ch, prefix + node[0] + "/"))
    return best


rng = random.Random(388)
for it in range(1500):
    roots = [gen_dir(rng, 0, 4) if rng.random() < 0.7 else ('f', rand_name(rng, True)) for _ in range(rng.randint(1, 3))]
    lines = []
    for r in roots:
        render(r, 0, lines)
    text = "\n".join(lines)
    expected = 0
    for r in roots:
        if r[0] == 'f':
            expected = max(expected, len(r[1]))
        else:
            expected = max(expected, oracle(r, ""))
    for f in all_methods:
        assert f(text) == expected, (f, text, expected, f(text))
print("OK: 1500 input ngẫu nhiên, 8 cách đều khớp oracle (duyệt cây thật)")

OK: 1500 input ngẫu nhiên, 8 cách đều khớp oracle (duyệt cây thật)


### 12.4. Kiểm tra hiệu năng ở cỡ lớn nhất

Chuỗi sâu nhất: một chuỗi thư mục lồng nhau rồi một file; và một chuỗi nhiều file phẳng. Cách 3/4 chạy tức thì.

In [23]:
import time

# độ dài ~10^4: thư mục lồng nhau độ sâu 100 mỗi tên "a"; rồi file
lines = []
for d in range(100):
    lines.append("\t" * d + "a")
lines.append("\t" * 100 + "f.g")
deep = "\n".join(lines)
print(len(deep) <= 10**4 * 1.2, len(deep))

wide = "d\n" + "\n".join("\tfile%d.txt" % i for i in range(700))
print(len(wide))

for name, f in [("stack", Solution().lengthLongestPath), ("levels", SolutionLevels().lengthLongestPath),
                ("names", SolutionNames().lengthLongestPath), ("brute", longest_path_brute)]:
    t0 = time.perf_counter()
    r1, r2 = f(deep), f(wide)
    print(f"{name:7} deep={r1} wide={r2}  {time.perf_counter() - t0:.4f}s")
assert Solution().lengthLongestPath(deep) == 100 * 2 + 3     # "a/" x100 + "f.g" = 200 + 3

True 5253
8991
stack   deep=203 wide=13  0.0002s
levels  deep=203 wide=13  0.0001s
names   deep=203 wide=13  0.0002s
brute   deep=203 wide=13  0.0095s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `+1` cho dấu `/`

In [24]:
def bug_no_slash(text):
    stack, best = [], 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        while len(stack) > d:
            stack.pop()
        cur = len(name) + (stack[-1] if stack else 0)        # SAI: thiếu + 1
        if '.' in name:
            best = max(best, cur)
        else:
            stack.append(cur)
    return best


print(bug_no_slash(examples[0][0]), "(đúng phải là 20)")

18 (đúng phải là 20)


**Vì sao sai:** đường dẫn nối các tên bằng `/` nên mỗi tầng thêm đúng một ký tự `/`. Ở ví dụ 1 thiếu 2 dấu `/` nên ra `18` thay vì `20`. **Sửa:** `cur = len(name) + stack[-1] + 1` khi stack không rỗng (hoặc lưu tiền tố đã kèm `/` như Cách 4).

### Lỗi 2: dùng `strip()` / `lstrip()` không đối số — mất dấu cách của tên

In [25]:
def bug_strip(text):
    stack, best = [], 0
    for line in text.split('\n'):
        name = line.lstrip()                                  # SAI: bỏ cả dấu cách đầu tên
        d = len(line) - len(name)
        while len(stack) > d:
            stack.pop()
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        if '.' in name:
            best = max(best, cur)
        else:
            stack.append(cur)
    return best


t = "a\n\t b.c"                  # file tên " b.c" (có dấu cách đầu) -> đường dẫn "a/ b.c" dài 6
print(bug_strip(t), "(đúng là", Solution().lengthLongestPath(t), ")")

5 (đúng là 6 )


**Vì sao sai:** `lstrip()` bỏ mọi khoảng trắng đầu dòng, kể cả dấu cách **thuộc về tên**; thế là tên ngắn đi và `d` cũng tính sai. **Sửa:** luôn `lstrip('\t')` với đối số rõ ràng.

### Lỗi 3: quên bỏ bớt ngăn xếp khi lên cấp

In [26]:
def bug_no_pop(text):
    stack, best = [], 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        cur = len(name) + (stack[-1] + 1 if stack else 0)      # SAI: không pop theo độ sâu
        if '.' in name:
            best = max(best, cur)
        else:
            stack.append(cur)
    return best


t = "a\n\tbbbbbbbbbb\nc.d"       # c.d ở gốc, không nằm trong 'bbbbbbbbbb'
print(bug_no_pop(t), "(đúng là", Solution().lengthLongestPath(t), ")")

16 (đúng là 3 )


**Vì sao sai:** sau khi đọc thư mục `bbbbbbbbbb` (con của `a`), file `c.d` ở gốc **không** thuộc `a` hay `bbbbbbbbbb`, nhưng stack vẫn giữ chúng nên cộng nhầm. **Sửa:** trước khi tính `cur`, `while len(stack) > d: stack.pop()`.

### Lỗi 4: cập nhật đáp án ở cả thư mục

In [27]:
def bug_dir_counts(text):
    stack, best = [], 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        while len(stack) > d:
            stack.pop()
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        best = max(best, cur)                                   # SAI: thư mục cũng được tính
        if '.' not in name:
            stack.append(cur)
    return best


print(bug_dir_counts("a"), "(đúng là", Solution().lengthLongestPath("a"), ")")
print(bug_dir_counts("aaaaaaaaaa\n\tb.c"), "(đúng là", Solution().lengthLongestPath("aaaaaaaaaa\n\tb.c"), ")")

1 (đúng là 0 )
14 (đúng là 14 )


**Vì sao sai:** đề chỉ hỏi đường dẫn tới **file**. Ví dụ 3 (`"a"`) phải trả `0`; thư mục không được tính. **Sửa:** chỉ `max` trong nhánh file (`'.' in name`).

## 14. Mẹo & nhận dạng pattern

- Đề cho **cây dưới dạng danh sách có thụt lề** (tab, dấu cách, dấu `-`...) ⇒ gần như luôn là bài **stack theo độ sâu** hoặc mảng theo độ sâu.
- "Duyệt DFS theo thứ tự" nghĩa là: tổ tiên của một dòng luôn là các dòng **trước đó** và gần nhất ở từng độ sâu. Đừng dựng cây rõ ràng nếu chỉ cần một con số tích lũy.
- Muốn tính *độ dài*, *tổng*, *đếm* đường dẫn? Giữ **số tích lũy** thay vì giữ cả đường dẫn (tiết kiệm `join`).
- Khi gặp dấu phân cách như `/`, nhớ công thức "k tên ⇒ k−1 dấu phân cách" (hoặc lưu tiền tố đã kèm dấu).
- Mẹo debug: in `repr(line)` để *nhìn thấy* các `\t` thay vì đoán.
- Pattern cùng họ: *Simplify Path* (stack theo `..`), *Flatten Nested List Iterator*, *Parse expression with indentation*, *Valid Parentheses* (stack khớp cặp).

## 15. Biến thể

1. **Trả về chính đường dẫn dài nhất** (chuỗi), thay vì độ dài ⇒ cần ngăn xếp tên (Cách 2) hoặc kèm vị trí.
2. **Đếm số file** hoặc **tổng độ dài mọi đường dẫn tới file**: dùng cùng stack, thay `max` bằng cộng dồn.
3. **Độ sâu lớn nhất** của cây: `max(d)` qua các dòng.
4. **Thụt lề bằng 4 dấu cách** hoặc số dấu cách tùy ý: độ sâu = `số dấu cách // 4` (hoặc dùng stack các mức thụt lề vì thụt lề không đều).
5. **Đường dẫn dài nhất tới bất kỳ mục nào** (cả thư mục): chỉ cần `max` ở mọi dòng.

Ví dụ biến thể 1 (trả về chuỗi đường dẫn):

In [28]:
def longest_path_string(text):
    path, best = [], ""
    for line in text.split('\n'):
        name = line.lstrip('\t')
        del path[len(line) - len(name):]
        if '.' in name:
            full = '/'.join(path + [name])
            if len(full) > len(best):
                best = full
        else:
            path.append(name)
    return best


print(longest_path_string(examples[1][0]))      # dir/subdir2/subsubdir2/file2.ext
print(repr(longest_path_string("a")))           # ''

dir/subdir2/subsubdir2/file2.ext
''


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute (lần ngược) | Mỗi file: quét ngược tìm tổ tiên rồi `join` | `O(n²)` | `O(n)` | Chỉ để hiểu đề / làm oracle |
| 2. Stack tên | Mô phỏng `cd`: giữ tên tổ tiên, `join` khi gặp file | `O(n²)` tệ nhất | `O(n)` | Cần **chính đường dẫn**, `n` nhỏ |
| 3. Stack độ dài | Giữ độ dài tích lũy mỗi tổ tiên | **`O(n)`** | `O(n)` | **Cách nộp chuẩn**, dễ giải thích |
| 4. Mảng theo độ sâu | `prefix[d]` ghi đè, không `pop` | **`O(n)`** | `O(D)` | Code ngắn nhất, hiểu "ghi đè an toàn" |

## 17. Bài tập vận dụng (dễ → khó)

### Bài 1 (đoán output)

Đọc đoạn code dưới và **đoán** nó in ra gì *trước khi chạy*. Điền vào `answer_1` (danh sách độ sâu).

In [29]:
def depths_of(text):
    return [len(l) - len(l.lstrip('\t')) for l in text.split('\n')]

sample_1 = "x\n\ty\n\t\tz\n\tw\nv"
# Đoán: depths_of(sample_1) in ra gì? (chưa gọi hàm - hãy tự đoán rồi gọi để kiểm tra)
answer_1 = None   # TODO: điền một list, ví dụ [0, 1, ...]

### Bài 2 (tính tay)

Với chuỗi `"a\n\tbb\n\t\tccc.d\n\tee.f\ng.hhhh"`, tính **bằng tay** độ dài đường dẫn dài nhất tới file (nhớ liệt kê từng file và đường dẫn của nó).

In [30]:
sample_2 = "a\n\tbb\n\t\tccc.d\n\tee.f\ng.hhhh"
answer_2 = None   # TODO: điền một số nguyên (đừng chạy hàm để lấy kết quả!)

### Bài 3 (điền TODO)

Hoàn thiện hàm theo Cách 3 (ngăn xếp độ dài). Chỗ `...` là phần bạn cần viết.

In [31]:
def length_longest_path_todo(text):
    stack = []
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        # TODO 1: bỏ bớt stack cho đến khi len(stack) == d
        ...
        # TODO 2: tính cur (độ dài đường dẫn tới mục hiện tại)
        cur = ...
        # TODO 3: file -> cập nhật best; thư mục -> push cur
        ...
    return best

### Bài 4 (tự viết lại)

Viết hàm `count_files(text)` trả về **số file** trong hệ thống file (không cần độ dài). Ví dụ `count_files("dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext")` phải bằng `2`.

In [32]:
def count_files(text):
    # TODO: tự viết
    pass

### Bài 5 (biến thể / mini challenge)

Viết `longest_path_any(text)` trả về độ dài đường dẫn dài nhất tới **bất kỳ mục nào** (file *hoặc* thư mục). Với `"a"` kết quả là `1`; với ví dụ 2 của đề là `32`. Gợi ý: cách duyệt giống Cách 3 nhưng `max` ở mọi dòng.

In [33]:
def longest_path_any(text):
    # TODO: tự viết
    pass

## 18. Gợi ý (3 mức)

**Bài 1**
- Mức 1: độ sâu của dòng = số tab ở đầu dòng.
- Mức 2: dòng `x` có 0 tab, `\ty` có 1 tab, `\t\tz` có 2 tab, `\tw` có 1 tab, `v` có 0 tab.
- Mức 3: đáp án là một list gồm 5 số.

**Bài 2**
- Mức 1: liệt kê mỗi file cùng các thư mục cha của nó (dựa vào độ sâu).
- Mức 2: `ccc.d` ở độ sâu 2 → cha là `bb` (độ sâu 1) → cha là `a`; `ee.f` ở độ sâu 1 → cha là `a`; `g.hhhh` ở gốc.
- Mức 3: `a/bb/ccc.d` = 1+1+2+1+5 = 10; `a/ee.f` = 1+1+4 = 6; `g.hhhh` = 6. Lấy lớn nhất.

**Bài 3**
- Mức 1: `while len(stack) > d: stack.pop()`.
- Mức 2: `cur = len(name) + (stack[-1] + 1 if stack else 0)`.
- Mức 3: `if '.' in name: best = max(best, cur)` `else: stack.append(cur)`.

**Bài 4**
- Mức 1: không cần stack. Mỗi dòng là file nếu có dấu chấm.
- Mức 2: `sum(1 for line in text.split('\n') if '.' in line)`.
- Mức 3: chú ý dấu chấm chỉ có trong tên file (đề cam kết), nên đếm dòng có `'.'`.

**Bài 5**
- Mức 1: sao chép Cách 3.
- Mức 2: thay vì chỉ `max` ở nhánh file, hãy `max` ngay sau khi tính `cur`.
- Mức 3: vẫn `push` khi là thư mục.

## 19. Đáp án tham khảo

In [34]:
# Bài 1
assert depths_of(sample_1) == [0, 1, 2, 1, 0]
answer_1 = [0, 1, 2, 1, 0]

# Bài 2: a/bb/ccc.d = 10, a/ee.f = 6, g.hhhh = 6 -> 10
answer_2 = 10
assert answer_2 == Solution().lengthLongestPath(sample_2) == 10


# Bài 3
def length_longest_path_todo(text):
    stack = []
    best = 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        while len(stack) > d:
            stack.pop()
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        if '.' in name:
            best = max(best, cur)
        else:
            stack.append(cur)
    return best


for text, expected in examples:
    assert length_longest_path_todo(text) == expected


# Bài 4
def count_files(text):
    return sum(1 for line in text.split('\n') if '.' in line)


assert count_files(examples[1][0]) == 2
assert count_files("a") == 0 and count_files("a.b\nc.d\ne") == 2


# Bài 5
def longest_path_any(text):
    stack, best = [], 0
    for line in text.split('\n'):
        name = line.lstrip('\t')
        d = len(line) - len(name)
        while len(stack) > d:
            stack.pop()
        cur = len(name) + (stack[-1] + 1 if stack else 0)
        best = max(best, cur)
        if '.' not in name:
            stack.append(cur)
    return best


assert longest_path_any("a") == 1
assert longest_path_any(examples[1][0]) == 32
assert longest_path_any("dddddddddd\n\te.f") == 14    # 10 + 1 + 3
print("Tất cả đáp án đã chạy đúng")

Tất cả đáp án đã chạy đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp độ dài)

```python
stack = []                      # stack[k] = độ dài đường dẫn tới tổ tiên độ sâu k
best = 0
for line in input.split('\n'):
    name = line.lstrip('\t')    # CHỈ bỏ tab
    d = len(line) - len(name)   # độ sâu = số tab
    while len(stack) > d:       # lùi ra khỏi thư mục sâu hơn
        stack.pop()
    cur = len(name) + (stack[-1] + 1 if stack else 0)   # +1 cho dấu '/'
    if '.' in name:             # file -> cập nhật đáp án
        best = max(best, cur)
    else:                       # thư mục -> làm tổ tiên cho các dòng sau
        stack.append(cur)
return best
```

### Checklist

- [ ] `lstrip('\t')` có đối số `'\t'` (không dùng `strip()` trơn).
- [ ] Độ sâu `d = len(line) - len(name)`; bỏ bớt stack cho đến khi `len(stack) == d`.
- [ ] Có `+ 1` cho dấu `/` khi stack không rỗng.
- [ ] Chỉ cập nhật `best` ở **file** (có `.`), thư mục thì `push`.
- [ ] Không có file ⇒ trả `0` (giá trị khởi tạo của `best`).
- [ ] Test: `"a"` → 0; file ở gốc `"a.b"` → 3; tên có dấu cách; hai file anh em; file sau khi lùi ra.